# Replogle – scIB-style Benchmarking with `embpy`

This notebook demonstrates the full `embpy` benchmarking workflow on the
**Replogle et al. (2022)** dataset — genome-scale CRISPRi Perturb-seq
in K562 cells.

We will:

1. **Load** the dataset from LaminDB via `embpy.pp.load_lamin`.
2. **Explore** the data structure.
3. **Run the scIB-style pipeline** (`embpy.tl.run_pipeline`) to generate
   protein embeddings, reduce them with PCA, train simple prediction models
   (Ridge, KNN, XGBoost), and compute evaluation metrics.
4. **Visualize** the results.
5. **Compute additional metrics** (phenocopy score, DEG overlap).

> **Requirements**: `pip install embpy lamindb`  
> You must be connected to the relevant LaminDB instance before running this notebook.

In [1]:
import logging

import matplotlib.pyplot as plt
import pandas as pd

import embpy

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("embpy")
logger.setLevel(logging.INFO)

## 1. Load the Replogle dataset from LaminDB

In [2]:
# Inspect the dataset card before downloading
card = embpy.pp.lamin_info("replogle")
print(f"Name:             {card.name}")
print(f"Artifact UID:     {card.uid}")
print(f"Description:      {card.description}")
print(f"Pert. column:     {card.perturbation_column}")
print(f"Pert. type:       {card.perturbation_type}")
print(f"Use case:         {card.use_case}")
print(f"Organism:         {card.organism}")
print(f"Reference:        {card.reference}")

Name:             replogle
Artifact UID:     cdTgT79SxGj3UJGy0000
Description:      Genome-scale CRISPRi Perturb-seq in K562 cells (Replogle et al., 2022).
Pert. column:     gene
Pert. type:       genetic
Use case:         protein
Organism:         human
Reference:        https://doi.org/10.1016/j.cell.2022.05.013


In [3]:
import lamindb as ln

ln.connect("theislab/pertmodeling")

→ defaulting to local storage: /ictstr01/groups/ml01/projects/big_perturbation/datasets
→ connected lamindb: theislab/pertmodeling
→ doing nothing, already connected lamindb: theislab/pertmodeling
→ defaulting to local storage: /ictstr01/groups/ml01/projects/big_perturbation/datasets
→ connected lamindb: theislab/pertmodeling


/home/icb/goncalo.pinto/.local/lib/python3.12/site-packages/lamindb_setup/_connect_instance.py:218: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  isinstance(v, types.ModuleType)
/home/icb/goncalo.pinto/.local/lib/python3.12/site-packages/lamindb_setup/_connect_instance.py:225: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  elif hasattr(v, "__module__") and getattr(v, "__module__", None):
/home/icb/goncalo.pinto/.local/lib/python3.12/site-packages/lamindb_setup/_connect_instance.py:226: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  class_module = v.__module__


In [4]:
# Download — wraps ln.Artifact.get("cdTgT79SxGj3UJGy0000").load()
adata = embpy.pp.load_lamin("replogle")
adata

→ doing nothing, already connected lamindb: theislab/pertmodeling
→ defaulting to local storage: /ictstr01/groups/ml01/projects/big_perturbation/datasets
→ connected lamindb: theislab/pertmodeling


INFO:embpy.pp.lamin_handler:Loading dataset 'replogle' (uid=cdTgT79SxGj3UJGy0000) from LaminDB …


! run input wasn't tracked, call `ln.track()` and re-run


AnnData object with n_obs × n_vars = 310385 × 8563
    obs: 'pert_target', 'pert_genetic', 'organism', 'cell_line', 'cell_type', 'disease', 'tissue_type', 'tissue', 'assay', 'suspension_type', 'donor_id', 'sex', 'self_reported_ethnicity', 'development_stage', 'pert_name', 'pert_type', 'batch', 'gene_id', 'transcript', 'gene_transcript', 'percent_mito', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'cancer', 'age', 'perturbation', 'ncounts', 'ngenes', 'nperts', 'percent_ribo'
    var: 'symbol', 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano', 'ncounts', 'ncells'
    uns: 'lamin_card'

## 2. Explore the data

In [5]:
print(f"Observations:  {adata.n_obs:,}")
print(f"Variables:     {adata.n_vars:,}")
print(f"\n.obs columns:  {list(adata.obs.columns)}")
print(f"\n.var columns:  {list(adata.var.columns)}")

Observations:  310,385
Variables:     8,563

.obs columns:  ['pert_target', 'pert_genetic', 'organism', 'cell_line', 'cell_type', 'disease', 'tissue_type', 'tissue', 'assay', 'suspension_type', 'donor_id', 'sex', 'self_reported_ethnicity', 'development_stage', 'pert_name', 'pert_type', 'batch', 'gene_id', 'transcript', 'gene_transcript', 'percent_mito', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'cancer', 'age', 'perturbation', 'ncounts', 'ngenes', 'nperts', 'percent_ribo']

.var columns:  ['symbol', 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano', 'ncounts', 'ncells']


In [6]:
pert_counts = adata.obs["pert_target"].value_counts()
print(f"Unique perturbations: {len(pert_counts)}")
print(f"\nTop 10 perturbations:")
pert_counts.head(10)

Unique perturbations: 2057

Top 10 perturbations:


pert_target
RPL3       1996
NCBP2       992
KIF11       974
SLC39A9     752
DONSON      745
NUP205      735
GAB2        674
BUB1        648
SNX15       637
CPSF3       627
Name: count, dtype: int64

In [7]:
adata.obs

,pert_target,pert_genetic,organism,cell_line,cell_type,disease,tissue_type,tissue,assay,suspension_type,...,z_gemgroup_UMI,core_scale_factor,core_adjusted_UMI_count,cancer,age,perturbation,ncounts,ngenes,nperts,percent_ribo
cell_barcode,,,,,,,,,,,,,,,,,,,,,
AAACCCAAGAAATCCA-27,NAF1,NAF1_+_164087918.23-P1P2|NAF1_-_164087674.23-P1P2,human,K-562,unknown,"chronic myelogenous leukemia, BCR-ABL1 positive",cell culture,bone marrow,10x 3' v3,cell,...,0.013047,0.813253,14064.5130,True,53,NAF1,11324.0,3332,1,0.225362
AAACCCAAGAACTTCC-31,BUB1,BUB1_-_111435363.23-P1P2|BUB1_-_111435372.23-P1P2,human,K-562,unknown,"chronic myelogenous leukemia, BCR-ABL1 positive",cell culture,bone marrow,10x 3' v3,cell,...,-1.522247,0.844107,6328.5845,True,53,BUB1,5257.0,2192,1,0.129732
AAACCCAAGAAGCCAC-34,UBL5,UBL5_+_9938801.23-P1P2|UBL5_-_9938639.23-P1P2,human,K-562,unknown,"chronic myelogenous leukemia, BCR-ABL1 positive",cell culture,bone marrow,10x 3' v3,cell,...,0.384157,1.091537,15853.7930,True,53,UBL5,17135.0,4002,1,0.236825
AAACCCAAGAATAGTC-43,C9orf16,C9orf16_+_130922603.23-P1P2|C9orf16_+_13092264...,human,K-562,unknown,"chronic myelogenous leukemia, BCR-ABL1 positive",cell culture,bone marrow,10x 3' v3,cell,...,3.721912,0.948277,31893.6200,True,53,C9orf16,29717.0,5358,1,0.246828
AAACCCAAGACAGCGT-28,TIMM9,TIMM9_-_58893848.23-P1P2|TIMM9_-_58893843.23-P1P2,human,K-562,unknown,"chronic myelogenous leukemia, BCR-ABL1 positive",cell culture,bone marrow,10x 3' v3,cell,...,-0.975371,0.868942,9674.9795,True,53,TIMM9,8261.0,2944,1,0.183392
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGTCTGTCGTC-45,ATP6V1D,ATP6V1D_+_67826485.23-P1P2|ATP6V1D_+_67826497....,human,K-562,unknown,"chronic myelogenous leukemia, BCR-ABL1 positive",cell culture,bone marrow,10x 3' v3,cell,...,0.428227,1.115052,16456.6390,True,53,ATP6V1D,18044.0,4300,1,0.253547
TTTGTTGTCTGTCTCG-27,CNOT3,CNOT3_-_54641691.23-P1P2|CNOT3_+_54641532.23-P1P2,human,K-562,unknown,"chronic myelogenous leukemia, BCR-ABL1 positive",cell culture,bone marrow,10x 3' v3,cell,...,-0.633593,0.813253,10662.1250,True,53,CNOT3,8510.0,3158,1,0.183196
TTTGTTGTCTGTGCGG-44,METTL3,METTL3_-_21979084.23-P1P2|METTL3_+_21979431.23...,human,K-562,unknown,"chronic myelogenous leukemia, BCR-ABL1 positive",cell culture,bone marrow,10x 3' v3,cell,...,1.054624,0.973352,21131.0960,True,53,METTL3,20355.0,4247,1,0.256595


## 3. Discover available embedding models

Replogle is a genetic perturbation dataset, so the relevant use case is
`"protein"`.  Let's see which embedding models are registered.

In [10]:
print(f"Use case: genetic")
print(f"\nAvailable embedding models:")
for m in embpy.tl.list_embedding_models("protein"):
    print(f"  - {m}")

Use case: genetic

Available embedding models:
  - esm2_8M
  - esm2_35M
  - esm2_150M
  - esm2_650M
  - esmc_300m
  - esmc_600m
  - prot_t5_xl


## 4. Run the scIB-style pipeline

We pick a subset of protein models for a quick run.  Change to
`embedding_models="all"` to benchmark every registered model.

In [ ]:
results = embpy.tl.run_pipeline(
    adata,
    perturbation_column=card.perturbation_column,
    use_case=card.use_case,
    embedding_models=["esm2_650M", "esmc_300m", "prot_t5_xl"],
    prediction_models=["ridge", "knn", "xgboost"],
    reduce_dim=50,
    mode="quick",
    test_size=0.2,
    random_state=42,
)

INFO:embpy.tl.pipeline:Pipeline: use_case='protein', 3 embedding models, 3 prediction models, mode='quick'
INFO:embpy.tl.pipeline:── Embedding model: esm2_650M
INFO:root:CUDA device found, using GPU.
INFO:root:GeneResolver initialized for human (Release 109).
INFO:root:Checking if Ensembl data needs downloading/indexing...
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /home/icb/goncalo.pinto/.cache/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /home/icb/goncalo.pinto/.cache/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /home/icb/goncalo.pinto/.cache/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.pep.all.fa.gz.pickle
INFO:root:pyensembl data is ready.
INFO:root:BioEmbedder initialized on device cuda, backend=api
INFO:root:Available models: ['atom_pair_count_fp', 'atom_pair_fp', 'bert_base_uncased', 'borzoi_

In [ ]:
results

NameError: name 'results' is not defined

### 4.1 Run all models

To benchmark every protein embedding model at once, simply pass `"all"`:

In [ ]:
# results_all = embpy.tl.run_pipeline(
#     adata,
#     perturbation_column=card.perturbation_column,
#     use_case=card.use_case,
#     embedding_models="all",
#     mode="quick",
# )
# results_all

### 4.2 Rigorous mode (cross-validated)

For publication-quality results, switch to `mode="rigorous"` which runs
5-fold cross-validation with randomized hyperparameter search:

In [ ]:
# results_rigorous = embpy.tl.run_pipeline(
#     adata,
#     perturbation_column=card.perturbation_column,
#     use_case=card.use_case,
#     embedding_models=["esm2_650M", "esmc_300m"],
#     mode="rigorous",
# )
# results_rigorous

## 5. Visualize results

The pipeline results DataFrame has one row per (embedding, prediction model)
combination.  We can use `embpy.pl` to visualize them.

In [ ]:
# Pivot to get one results-DataFrame per embedding (index = prediction model)
results_by_embedding = {}
for emb_name, grp in results.groupby("embedding"):
    df = grp.set_index("model").drop(columns=["embedding", "time_s"], errors="ignore")
    results_by_embedding[emb_name] = df

print("Embeddings:", list(results_by_embedding.keys()))

In [ ]:
# Compare embeddings side by side
fig = embpy.pl.plot_benchmark_comparison(
    results_by_embedding,
    metrics=["r2", "pearson", "spearman"],
)
plt.show()

In [ ]:
# Per-embedding breakdown
for emb_name, df in results_by_embedding.items():
    fig = embpy.pl.plot_benchmark(df, title=f"Replogle — {emb_name}")
    plt.show()

### 5.1 Custom ranking table

Rank embedding × model combinations by a metric of choice.

In [ ]:
ranking = (
    results
    .sort_values("pearson", ascending=False)
    .reset_index(drop=True)
    [["embedding", "model", "r2", "pearson", "spearman", "mse", "time_s"]]
)
ranking

### 5.2 Timing comparison

In [ ]:
timing = (
    results
    .groupby("embedding")["time_s"]
    .first()
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(8, 4))
timing.plot.barh(ax=ax, color="steelblue", edgecolor="k")
ax.set_xlabel("Time (seconds)")
ax.set_title("Replogle — Embedding computation time")
fig.tight_layout()
plt.show()

## 6. Individual benchmark (single embedding)

If you only need to evaluate one embedding at a time, use
`embpy.tl.benchmark_embeddings` directly.

In [ ]:
single_result = embpy.tl.benchmark_embeddings(
    adata,
    perturbation_column=card.perturbation_column,
    perturbation_type=card.perturbation_type,
    embedding_model="esm2_650M",
    models=["ridge", "knn"],
    mode="quick",
    reduce_dim=50,
)
single_result

In [ ]:
fig = embpy.pl.plot_benchmark(single_result, title="Replogle — ESM2-650M")
plt.show()

## 7. Additional metrics

Beyond the regression metrics computed by the pipeline, `embpy.tl` provides
biological evaluation tools.  These require a **predicted** AnnData
(`adata_pred`) alongside the real one — for example, the output of a
perturbation prediction model (GEARS, scGEN, CPA, etc.).

### 7.1 cell-eval (ArcInstitute)

`embpy.tl.cell_eval` wraps the [cell-eval](https://github.com/ArcInstitute/cell-eval)
evaluation suite.  It computes differential-expression-based and
AnnData-pair metrics comparing predicted vs. real single-cell perturbation
data.

Profiles available: `"full"`, `"vcc"`, `"minimal"`, `"de"`, `"anndata"`.

> Requires: `pip install cell-eval`

In [ ]:
# --- Option A: pipeline-style wrapper (recommended) ---
# Returns a tidy DataFrame + stores aggregate in adata.uns["cell_eval_agg"]
#
# cell_eval_results = embpy.tl.run_cell_eval(
#     adata_pred=adata_pred,
#     adata_real=adata,
#     control_pert="control",
#     pert_col=card.perturbation_column,
#     profile="full",          # or "minimal", "de", "vcc", "anndata"
#     num_threads=4,
# )
# display(cell_eval_results.head())
# display(adata.uns["cell_eval_agg"])

# --- Option B: low-level wrapper (returns both DataFrames) ---
#
# results_per_pert, results_agg = embpy.tl.cell_eval(
#     adata_pred=adata_pred,
#     adata_real=adata,
#     control_pert="control",
#     pert_col=card.perturbation_column,
#     profile="full",
#     num_threads=4,
# )
# display(results_agg)

### 7.2 Phenocopy score (PRESAGE-inspired)

The phenocopy score evaluates whether predicted expression profiles preserve
the functional similarity structure of real perturbations.  It computes
cosine-similarity matrices, applies MAD-based thresholds, and reports
AUROC and Recall@k.

In [ ]:
# scores = embpy.tl.phenocopy_score_adata(
#     adata_true=adata,
#     adata_pred=adata_pred,
#     pert_col=card.perturbation_column,
#     n_components=50,
#     k=5,
# )
# print(f"AUROC:     {scores['auroc']:.3f}")
# print(f"Recall@5:  {scores['recall_at_k']:.3f}")

### 7.3 DEG overlap

Compare differentially expressed gene lists between predicted and real
perturbation profiles.

In [ ]:
# Compare DEGs between real and predicted (requires scanpy):
#
# deg_results = embpy.tl.compare_deg(
#     adata_true=adata,
#     adata_pred=adata_pred,
#     group_col=card.perturbation_column,
#     top_n=50,
# )
# print(f"Jaccard overlap:      {deg_results['jaccard']:.3f}")
# print(f"Direction agreement:  {deg_results['direction_agreement']:.3f}")

## 8. Accessing stored results

The pipeline automatically stores results in `adata.uns["pipeline_results"]`
for downstream use.

In [ ]:
stored = adata.uns["pipeline_results"]
print(f"Stored results shape: {stored.shape}")
stored.head()

## Summary

| Step | Function | What it does |
|---|---|---|
| Load | `embpy.pp.load_lamin("replogle")` | Fetch dataset from LaminDB by name |
| Discover | `embpy.tl.list_embedding_models("protein")` | List available models for the use case |
| Benchmark | `embpy.tl.run_pipeline(...)` | Full scIB-style pipeline (embed → PCA → train → evaluate) |
| Benchmark (single) | `embpy.tl.benchmark_embeddings(...)` | Evaluate one embedding model |
| Visualize | `embpy.pl.plot_benchmark_comparison(...)` | Side-by-side embedding comparison |
| Visualize | `embpy.pl.plot_benchmark(...)` | Per-embedding metric bars |
| Metrics | `embpy.tl.cell_eval(...)` | ArcInstitute cell-eval suite (DE, VCC, AnnData metrics) |
| Metrics | `embpy.tl.phenocopy_score_adata(...)` | PRESAGE-inspired phenocopy AUROC & recall |
| Metrics | `embpy.tl.compare_deg(...)` | DEG overlap between real and predicted |